## Boltz 2

Accurately modeling biomolecular interactions is a central challenge in modern biology. While
recent advances, such as AlphaFold3 and Boltz-1, have substantially improved our ability to predict biomolecular complex structures, these models still fall short in predicting binding affinity, a
critical property underlying molecular function and therapeutic efficacy. Here, we present Boltz-2,
a new structural biology foundation model that exhibits strong performance for both structure and
affinity prediction. Boltz-2 introduces controllability features including experimental method conditioning, distance constraints, and multi-chain template integration for structure prediction, and
is, to our knowledge, the first AI model to approach the performance of free-energy perturbation
(FEP) methods in estimating small molecule–protein binding affinity. Crucially, it achieves strong
correlation with experimental readouts on many benchmarks, while being at least 1000× more computationally efficient than FEP. By coupling Boltz-2 with a generative model for small molecules,
we demonstrate an effective workflow to find diverse, synthesizable, high-affinity binders, as estimated by absolute FEP simulations on the TYK2 target. To foster broad adoption and further
innovation at the intersection of machine learning and biology, we are releasing Boltz-2 weights,
inference, and training code 1 under a permissive open license, providing a robust and extensible
foundation for both academic and industrial research.

Paper: http://jeremywohlwend.com/assets/boltz2.pdf  
Github: https://github.com/jwohlwend/boltz  
Docs: https://github.com/jwohlwend/boltz/blob/main/docs/prediction.md   

Important functionality updates over Boltz 1:
1. Affinity prediction for Small Molecule ligands
2. Templates
3. Contact constraints (was only pocket constraints before)

In [11]:
import biolib

In [12]:
biolib.login()

2025-06-09 05:00:39,655 | INFO : Already signed in


In [16]:
boltz = biolib.load('@nn/DCD/Boltz-2:0.0.37')

2025-06-09 05:06:06,971 | INFO : Loaded project @nn/DCD/Boltz-2:0.0.37


### Running the tool
Boltz-2 takes a YAML file as input format.  
Please make sure to read the docs on how to structure the YAML: https://github.com/jwohlwend/boltz/blob/main/docs/prediction.md  
When no `msa` entry is given for a protein, the tool will automatically generate an MSA. 


In the cell below, a YAML file is written to disk. You can also use your own YAML file.

In [9]:
%%writefile input.yaml
version: 1
sequences:
  - protein:
      id: [A]
      sequence: MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDAL
  - ligand:
      id: [C, D]
      ccd: SAH
  - ligand:
      id: [E, F]
      smiles: N[C@@H](Cc1ccc(O)cc1)C(=O)O

Overwriting input.yaml


In [14]:
job = boltz.run(
    input='input.yaml',
    recycling_steps='3', # Increase for larger complexes
    diffusion_samples='1', 
    sampling_steps='200'
)

2025-06-09 05:00:45,705 | INFO : View the result in your browser at: https://biolib.corp.novocorp.net/results/59f66543-aeff-4440-ab9b-5f9b598087c3/
2025-06-09 05:00:47,075 | INFO : Cloud: Initializing
2025-06-09 05:00:49,155 | INFO : Cloud: Pulling images...
2025-06-09 05:00:51,233 | INFO : Cloud: Computing...
Analyzing input files...
Using the following YAML input:
sequences:
- protein:
    id:
    - A
    sequence: MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDAL
- ligand:
    ccd: SAH
    id:
    - C
    - D
- ligand:
    id:
    - E
    - F
    smiles: N[C@@H](Cc1ccc(O)cc1)C(=O)O
version: 1

Initializing boltz-2...
Loading model...
Checking input data.
Processing 1 inputs with 1 threads.
  0% 0/1 [00:00<?, ?it/s]Generating MSA for boltz.yaml with 1 protein entities.

  0% 0/150 [00:00<?, ?it/s]
SUBMIT:   0% 0/150 [00:00<?, ?it/s]
COMPLETE: 100% 150/150 [00:00<00:00, 12745.80it/s]
100% 1/1 [00:00<00:00,  2.57it/s]
Using bfloat16 Automatic Mixed P

In [ ]:
# Your .cif files will be in the `predictions/boltz` folder
job.save_files('boltz-output')

#### Affinity Prediction

In [7]:
%%writefile affinity.yaml
version: 1  # Optional, defaults to 1
sequences:
  - protein:
      id: A
      sequence: MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ
  - ligand:
      id: B
      smiles: 'N[C@@H](Cc1ccc(O)cc1)C(=O)O'
properties:
  - affinity:
      binder: B

Writing affinity.yaml


In [8]:
job = boltz.run(
    input='affinity.yaml',
    diffusion_samples='1', 
    sampling_steps='200'
)

2025-06-08 19:43:59,853 | INFO : View the result in your browser at: https://biolib.corp.novocorp.net/results/6db64898-3587-4641-80f9-01e125929cf4/
2025-06-08 19:44:01,207 | INFO : Cloud: Initializing
2025-06-08 19:44:03,283 | INFO : Cloud: Pulling images...
2025-06-08 19:44:05,363 | INFO : Cloud: Computing...
Analyzing input files...
Using the following YAML input:
properties:
- affinity:
    binder: B
sequences:
- protein:
    id: A
    sequence: MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ
- ligand:
    id: B
    smiles: N[C@@H](Cc1ccc(O)cc1)C(=O)O
version: 1

Initializing boltz-2...
Loading model...
Checking input data.
Processing 1 inputs with 1 t

In [9]:
import json

affinity_file = job.get_output_file('predictions/boltz/affinity_boltz.json')
print(json.load(affinity_file.get_file_handle()))

{'affinity_pred_value': 2.608443260192871, 'affinity_probability_binary': 0.4031522870063782, 'affinity_pred_value1': 2.792787551879883, 'affinity_probability_binary1': 0.3709689974784851, 'affinity_pred_value2': 2.4240987300872803, 'affinity_probability_binary2': 0.43533554673194885}


#### Using Templates
Templates can be passed as a root level element in the YAML with the following configurations:
```
templates:
    - cif: CIF_PATH  # if only a path is provided, Boltz will find the best matchings
    - cif: CIF_PATH
      chain_id: CHAIN_ID   # optional, specifiy which chain to find a template for
    - cif: CIF_PATH
      chain_id: [CHAIN_ID, CHAIN_ID]  # can be more than one
      template_id: [TEMPLATE_CHAIN_ID, TEMPLATE_CHAIN_ID]
```

In [31]:
%%writefile templates.yaml
version: 1
sequences:
  - protein:
      id: [A]
      sequence: MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG
templates:
    - cif: templates/1ubq.cif
      chain_id: [A]

Overwriting templates.yaml


In [15]:
job = boltz.run(
    input='templates.yaml',
    diffusion_samples='1', 
    sampling_steps='200',
    templates='templates/'
)

2025-06-09 05:03:02,082 | INFO : View the result in your browser at: https://biolib.corp.novocorp.net/results/2c66c27e-26e1-45dd-b380-95206aeb30a1/
2025-06-09 05:03:03,432 | INFO : Cloud: Initializing
2025-06-09 05:03:05,512 | INFO : Cloud: Pulling images...
2025-06-09 05:03:07,588 | INFO : Cloud: Computing...
Analyzing input files...
Using the following YAML input:
sequences:
- protein:
    id:
    - A
    sequence: MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG
templates:
- chain_id:
  - A
  cif: templates/1ubq.cif
version: 1

Initializing boltz-2...
Loading model...
Checking input data.
Processing 1 inputs with 1 threads.
  0% 0/1 [00:00<?, ?it/s]Generating MSA for boltz.yaml with 1 protein entities.

  0% 0/150 [00:00<?, ?it/s]
SUBMIT:   0% 0/150 [00:00<?, ?it/s]
COMPLETE: 100% 150/150 [00:00<00:00, 10703.21it/s]
100% 1/1 [00:00<00:00,  2.19it/s]
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False,

#### Using constraints
You can provide constraints on what should bind (be a certain distance from eachother) in the structure prediction.  
The constraints can take the following forms:
```
constraints:
    - bond:
        atom1: [CHAIN_ID, RES_IDX, ATOM_NAME]
        atom2: [CHAIN_ID, RES_IDX, ATOM_NAME]
    - pocket:
        binder: CHAIN_ID
        contacts: [[CHAIN_ID, RES_IDX/ATOM_NAME], [CHAIN_ID, RES_IDX/ATOM_NAME]]
        max_distance: DIST_ANGSTROM
    - contact:
        token1: [CHAIN_ID, RES_IDX/ATOM_NAME]
        token2: [CHAIN_ID, RES_IDX/ATOM_NAME]
        max_distance: DIST_ANGSTROM
```

This example will show the contact constraints new to Boltz-2 on some sample data

In [16]:
%%writefile constraints.yaml
version: 1
sequences:
  - protein:
      id: [A]
      sequence: MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDAL
  - protein:
      id: [B]
      sequence: MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDAL
constraints:
  - contact:
      token1: [A, 10]
      token2: [B, 10]
      max_distance: 4.0

Writing constraints.yaml


In [18]:
job = boltz.run(
    input='constraints.yaml',
    diffusion_samples='1', 
    sampling_steps='200',
)

2025-06-08 20:02:04,223 | INFO : View the result in your browser at: https://biolib.corp.novocorp.net/results/d15d9ff6-4be4-456a-a427-b6b5df2b25d2/
2025-06-08 20:02:05,500 | INFO : Cloud: The job has been queued. Please wait...
2025-06-08 20:02:07,783 | INFO : Cloud: The job has been queued. Please wait...
2025-06-08 20:02:11,076 | INFO : Cloud: The job has been queued. Please wait...
2025-06-08 20:02:15,374 | INFO : Cloud: The job has been queued. Please wait...
2025-06-08 20:02:20,735 | INFO : Cloud: Initializing
2025-06-08 20:02:20,735 | INFO : Cloud: Pulling images...
2025-06-08 20:02:22,811 | INFO : Cloud: Computing...
Analyzing input files...
Using the following YAML input:
properties:
- affinity:
    binder: B
sequences:
- protein:
    id: A
    sequence: MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVE

##### Using custom MSAs
To use custom MSAs, you can specify their filename in YAML and provide them via the `biolib_files` argument in `biolib.run()`.  
Be aware, that you can only use one MSA per protein and it has to be in `.a3m` format.

In [ ]:
%%writefile input_custom_msa.yaml
version: 1
sequences:
  - protein:
      id: [A]
      sequence: MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDAL
      msa: custom.a3m
  - ligand:
      id: [C, D]
      ccd: SAH
  - ligand:
      id: [E, F]
      smiles: N[C@@H](Cc1ccc(O)cc1)C(=O)O

In [ ]:
job = boltz.run(
    input='input_custom_msa.yaml',
    recycling_steps='3', # Increase for larger complexes
    diffusion_samples='1', 
    sampling_steps='200',
    biolib_files=['custom.a3m'] # Files in this list will be made available to the application at runtime
)

In [ ]:
# Your .cif files will be in the `predictions/boltz` folder
job.save_files('boltz-output')